In [ ]:
!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 trl==0.4.7 tokenizers==0.13.3 sentencepiece tensorboard

In [ ]:
!git pull

In [ ]:
!rm -rf results

In [ ]:
import os
import torch
from datasets import load_dataset
from datasets.arrow_dataset import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

import pandas as pd
import pyarrow as pa

import glob

from datetime import datetime

import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

os.environ['HF_TOKEN'] = token

In [2]:
size = "13"

# The model that you want to train from the Hugging Face hub
model_name = f"NousResearch/llama-2-{size}b-chat-hf"

# Fine-tuned model name
new_model = f"llama-2-{size}b-generate-questions"

################################################################################
# QLoRA parameters
################################################################################

# LoRA attention dimension
lora_r = 512 # 64 before

# Alpha parameter for LoRA scaling
lora_alpha = 16

# Dropout probability for LoRA layers
lora_dropout = 0.1 # 0.1 before

################################################################################
# bitsandbytes parameters
################################################################################

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = False

################################################################################
# TrainingArguments parameters
################################################################################

# Output directory where the model predictions and checkpoints will be stored
output_dir = "./results"

# Number of training epochs
num_train_epochs = 18

# Enable fp16/bf16 training (set bf16 to True with an A100)
fp16 = False
bf16 = False

# Batch size per GPU for training
per_device_train_batch_size = 5

# Batch size per GPU for evaluation
per_device_eval_batch_size = 2

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 1

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate (AdamW optimizer)
learning_rate = 2e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer to use
optim = "paged_adamw_32bit"

# Learning rate schedule (constant a bit better than cosine)
lr_scheduler_type = "constant"

# Number of training steps (overrides num_train_epochs)
max_steps = -1

# Ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

# Group sequences into batches with same length
# Saves memory and speeds up training considerably
group_by_length = True

# Save checkpoint every X updates steps
save_steps = 1000

# Log every X updates steps
logging_steps = 25

################################################################################
# SFT parameters
################################################################################

# Maximum sequence length to use
max_seq_length = None

# Pack multiple short examples in the same input sequence to increase efficiency
packing = False

# Load the entire model on the GPU 0
device_map = {"": 0}

file_to_save_predictions = str(datetime.now()) + ".txt"

TEMPERATURE = 0.3
MAX_LENGTH = 1400

### Preparing the dataset to be used

In [3]:
def generate_question(path_html):

    prompt = \
f"""Gere o esqueleto de uma atividade para auxiliar o aluno no aprendizado.
O esqueleto da atividade deverá conter a estrutura em HTML.
Se a questão contiver imagens, adicione uma descrição no atributo de alt.
Adicione também uma explicação para a estrutura HTML em um novo atributo exp quando necessário.
Quando for necessário utilizar CSS, adicione uma explicação na forma de comentários, acima da classe. 

Utilize os seguintes parâmetros:
"""

    with open(path_html, "r") as file:
        html_text_list = file.readlines()

    while ':' in html_text_list[0]:
        prompt += html_text_list[0]
        html_text_list.pop(0)

    html_text = "".join(html_text_list)
    
    prompt += \
"""
Siga as seguintes regras:

Gere uma descrição textual da atividade e em seguida o html contendo a questão.
Não adicione nada fora a descrição textual e o html da questão.
Gere uma atividade que seja criativa e que ajude o aluno a aprender de maneira lúdica.
"""


    text = f"""<s>[INST]\n{prompt}[/INST]\n{html_text}</s>"""

    return [path_html, text]

def generate_questions(path_htmls):

    lines = []
    for html in glob.iglob(f"{path_htmls}/*.html", recursive = True):
        if html.endswith("base.html"):
            continue

        lines.append(
            generate_question(html)
        )

    return pd.DataFrame(lines, columns = ['file','text'])

In [4]:
data = generate_questions(
    "handmade_html_dataset/**"
)
data

,file,text
0,handmade_html_dataset/matemática/contagem/exa...,<s>[INST]\nGere o esqueleto de uma atividade p...
1,handmade_html_dataset/matemática/contagem/exa...,<s>[INST]\nGere o esqueleto de uma atividade p...
2,handmade_html_dataset/matemática/contagem/exa...,<s>[INST]\nGere o esqueleto de uma atividade p...
3,handmade_html_dataset/matemática/contagem/exa...,<s>[INST]\nGere o esqueleto de uma atividade p...
4,handmade_html_dataset/matemática/contagem/exa...,<s>[INST]\nGere o esqueleto de uma atividade p...
5,handmade_html_dataset/matemática/contagem/exa...,<s>[INST]\nGere o esqueleto de uma atividade p...
6,handmade_html_dataset/matemática/contagem/exa...,<s>[INST]\nGere o esqueleto de uma atividade p...
7,handmade_html_dataset/matemática/contagem/exa...,<s>[INST]\nGere o esqueleto de uma atividade p...
8,handmade_html_dataset/matemática/contagem/exa...,<s>[INST]\nGere o esqueleto de uma atividade p...
9,handmade_html_dataset/matemática/adição/exa...,<s>[INST]\nGere o esqueleto de uma atividade p...


In [5]:
print(data.iloc[23,:]['text'])

<s>[INST]
Gere o esqueleto de uma atividade para auxiliar o aluno no aprendizado.
O esqueleto da atividade deverá conter a estrutura em HTML.
Se a questão contiver imagens, adicione uma descrição no atributo de alt.
Adicione também uma explicação para a estrutura HTML em um novo atributo exp quando necessário.
Quando for necessário utilizar CSS, adicione uma explicação na forma de comentários, acima da classe. 

Utilize os seguintes parâmetros:
nivel: 2 ano fundamental 
assunto: adição, contagem 
tematica: piratas
largura-folha: 600px 
layout: várias imagens, ligar imagens

Siga as seguintes regras:

Gere uma descrição textual da atividade e em seguida o html contendo a questão.
Não adicione nada fora a descrição textual e o html da questão.
Gere uma atividade que seja criativa e que ajude o aluno a aprender de maneira lúdica.
[/INST]

Nessa questão serão trabalhados os princípios da adição e
contagem. Essa questão consiste de uma imagem de um pirata, que já tem 15 moedas
de ouro, e ma

In [6]:
dataset_name = "questions"
dataset = Dataset(pa.Table.from_pandas(data[['text']]))
dataset

Dataset({
    features: ['text'],
    num_rows: 30
})

In [7]:
# Load tokenizer and model with QLoRA configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix weird overflow issue with fp16 training

# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)

# Set training parameters
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard"
)

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=packing,
)

# Train model
trainer.train()

# Save trained model
trainer.model.save_pretrained(new_model)

Your GPU supports bfloat16: accelerate training with bf16=True


/home/aipim/anaconda3/envs/mestrado-matheus/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/home/aipim/anaconda3/envs/mestrado-matheus/lib/python3.9/site-packages/peft/utils/other.py:102: FutureWarning: prepare_model_for_int8_training is deprecated and will be removed in a future version. Use prepare_model_for_kbit_training instead.
  warnings.warn(
/home/aipim/anaconda3/envs/mestrado-matheus/lib/python3.9/site-packages/trl/trainer/sft_trainer.py:159: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

/home/aipim/anaconda3/envs/mestrado-matheus/lib/python3.9/site-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss
25,1.242500
50,0.413800
75,0.215800
100,0.090300


## Generation of some questions to verify the quality

In [8]:
# Ignore warnings
logging.set_verbosity(logging.CRITICAL)

# Run text generation pipeline with our next model
prompts = [
"""
nivel: 2 ano fundamental
assunto: adição
tematica: lanches
largura-folha: 700px
layout: única imagem
""",
"""
nivel: 2 ano fundamental
assunto: adição
tematica: lanches
largura-folha: 700px
layout: duas imagens, linhas e colunas
""",

"""
nivel: 2º ano fundamental
assunto: adição
tematica: carrinhos
largura-folha: 600px
layout: única imagem
""",
"""
nivel: 2º ano fundamental
assunto: adição
tematica: carrinhos
largura-folha: 600px
layout: três imagens
""",

"""
nivel: 2º ano fundamental
assunto: adição
tematica: fundo do mar
largura-folha: 600px
layout: única imagem
""",
"""
nivel: 2º ano fundamental
assunto: adição
tematica: fundo do mar
largura-folha: 600px
layout: três imagens
""",

"""
nivel: 2º ano fundamental
assunto: subtração
tematica: fogos de artifício
largura-folha: 800px
layout: única imagem
""",
"""
nivel: 2º ano fundamental
assunto: subtração
tematica: fogos de artifício
largura-folha: 800px
layout: três imagens
""",
]

pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length = MAX_LENGTH, temperature = TEMPERATURE)

for i,p in enumerate(prompts):
	inp = \
f"""<s>
[INST]
Gere o esqueleto de uma atividade para auxiliar o aluno no aprendizado.
O esqueleto da atividade deverá conter a estrutura em HTML.
Se a questão contiver imagens, adicione uma descrição no atributo de alt.
Adicione também uma explicação para a estrutura HTML em um novo atributo exp quando necessário.
Quando for necessário utilizar CSS, adicione uma explicação na forma de comentários, acima da classe. 

Utilize os seguintes parâmetros:
{p}

Siga as seguintes regras:

Gere uma descrição textual da atividade e em seguida o html contendo a questão.
Não adicione nada fora a descrição textual e o html da questão.
Gere uma atividade que seja criativa e que ajude o aluno a aprender de maneira lúdica.
[/INST]"""

	result = pipe(inp)[0]['generated_text']

	with open(f"runs/{file_to_save_predictions}","a",encoding='utf-8') as file:
		file.write("\n---------------------------------------------------------------------------------------------------------------\n")
		file.write(result)

	print(f"done {i}")
	

/home/aipim/anaconda3/envs/mestrado-matheus/lib/python3.9/site-packages/transformers/generation/utils.py:1270: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use a generation configuration file (see https://huggingface.co/docs/transformers/main_classes/text_generation )
  warnings.warn(
/home/aipim/anaconda3/envs/mestrado-matheus/lib/python3.9/site-packages/torch/utils/checkpoint.py:91: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


done 0
done 1
done 2
done 3
done 4
done 5
done 6
done 7


In [9]:
line = [
    model_name,
	lora_r, 
	lora_alpha, 
	lora_dropout, 
	num_train_epochs, 
	per_device_train_batch_size, 
	per_device_eval_batch_size, 
	gradient_accumulation_steps, 
	max_grad_norm, learning_rate, 
	weight_decay, 
	optim, 
	file_to_save_predictions
]
if os.path.exists("runs/parameters_train.csv"):
    dataFrame = pd.read_csv("runs/parameters_train.csv")
    dataFrame = pd.concat([dataFrame,pd.DataFrame([line],columns=dataFrame.columns)],ignore_index=True)
else:
    dataFrame = pd.DataFrame([line],columns = ["model_name","lora_r", "lora_alpha", "lora_dropout", "num_train_epochs", "per_device_train_batch_size", "per_device_eval_batch_size", "gradient_accumulation_steps", "max_grad_norm", "learning_rate", "weight_decay", "optim", "file_to_save_predictions"])
dataFrame.to_csv("runs/parameters_train.csv",index=False)
    

In [ ]:
!git status

In [ ]:
!git add .

In [ ]:
!git commit -m "adding experiments"

In [ ]:
!git push